# 把 `nnx.split` 的 params 存成 `.npz` 并加载回模型

本 notebook 演示一种**不依赖 pickle** 的参数持久化方式：把 `flax.nnx` 模型的参数逐叶子写进 `.npz`，之后能原样读回并 `nnx.merge` 进模型。

结论先说：**可以**。`nnx.split(model)` 得到的 `state` 是 `flax.nnx.statelib.State`——普通的 pytree，叶子就是 jax 数组，天然适合塞进 npz。但要无损还原，除了数组本身还必须额外保存两项**结构元信息**：

| npz 键 | 内容 | 为什么需要 |
|---|---|---|
| `p0000`, `p0001`, … | 第 i 个叶子的数组 | 参数本体 |
| `__order__` | 每个叶子的可读路径字符串 | `npz` 内部按键名**排序**保存，不记录顺序就无法按原顺序还原 |
| `__treedef__` | `pickle.dumps(PyTreeDef)` 的字节 | `jax.tree_util.tree_unflatten` 靠它重建同样的容器结构（`State` 的嵌套字典/`nnx.List` 等） |

`__treedef__` 只存**结构**、不存任何数值，因此它很小（几 KB）；参数值全在明文数组里，可以脱离 Python 环境被读（例如 `numpy`、MATLAB、其它语言）。

本仓库中的实际用法见 `NES-VMC K4/H2_631G_8Qbit_K4_Ansatz_A_RealFFNN.py` 与 `NES-VMC K6/…K6….py` 里的同名函数 `save_params_npz` / `load_params_npz`。

> 下面示例中的模型用 `param_dtype=float` 且开启 x64，所以叶子是 `float64`，与仓库里的 K4/K6 脚本一致。

In [1]:
import pickle
import numpy as np

import jax
import jax.numpy as jnp
import flax.nnx as nnx

# 必须！否则 JAX 会把 float64 静默降为 float32（细节见最后一节）
jax.config.update("jax_enable_x64", True)

print("jax      :", jax.__version__)
print("flax     :", __import__("flax").__version__)
print("numpy    :", np.__version__)
print("x64 开启 :", jax.config.jax_enable_x64)

jax      : 0.9.2
flax     : 0.12.6
numpy    : 2.4.6
x64 开启 : True


## 1. 一个最小的 nnx 模型

结构不重要，这里故意做成和使用场景相近的形态：**一个容器模块 + `nnx.List` 里放多个子模块**，这样后面 `State` 的路径不会被拍平，能演示真实的嵌套情况。

In [2]:
class MLP(nnx.Module):
    """两层 tanh MLP + 两个输出头（实参数）。"""

    def __init__(self, n_in: int, hidden: int = 8, *, rngs: nnx.Rngs):
        super().__init__()
        # param_dtype=float + x64 → float64，与仓库里的模型保持一致
        self.backbone1 = nnx.Linear(n_in, hidden, rngs=rngs, param_dtype=float)
        self.backbone2 = nnx.Linear(hidden, hidden, rngs=rngs, param_dtype=float)
        self.head_u = nnx.Linear(hidden, 1, rngs=rngs, param_dtype=float)
        self.head_v = nnx.Linear(hidden, 1, rngs=rngs, param_dtype=float)

    def __call__(self, x):
        h = nnx.tanh(self.backbone1(x))
        h = nnx.tanh(self.backbone2(h))
        return jnp.squeeze(self.head_u(h)), jnp.squeeze(self.head_v(h))


class Ensemble(nnx.Module):
    """把 K 个 MLP 放进 nnx.List，模拟 NES 的 K 个单态拟设。"""

    def __init__(self, K: int, n_in: int, hidden: int = 8, *, rngs: nnx.Rngs):
        super().__init__()
        self.K = K
        self.mlps = nnx.List()
        key = rngs.params()
        for _ in range(K):
            key, sub = jax.random.split(key)
            self.mlps.append(MLP(n_in, hidden, rngs=nnx.Rngs(params=sub)))

    def __call__(self, x):
        return jnp.stack([m(x)[0] for m in self.mlps], axis=-1)


model = Ensemble(K=3, n_in=4, hidden=8, rngs=nnx.Rngs(0))
x = jnp.linspace(-1.0, 1.0, 5 * 4).reshape(5, 4)
y_ref = model(x)
print("输入 :", x.shape)
print("输出 :", y_ref.shape)
print(y_ref)

输入 : (5, 4)
输出 : (5, 3)
[[-0.24103521 -0.17351907  0.38299947]
 [-0.18949238 -0.14880035  0.36389023]
 [-0.08921869 -0.04138292  0.17255699]
 [ 0.09868382  0.03211686 -0.09456598]
 [ 0.20680205  0.03282466 -0.21687871]]


## 2. `nnx.split`：拆成「结构」与「参数」

`nnx.split(model)` 返回 `(graphdef, state)`：

* `graphdef`：静态结构（层类型、维度、连接关系），不含数值；
* `state`：`flax.nnx.statelib.State`，是一棵叶子为 jax 数组的 pytree。

注意 `nnx.State` 的叶子**外面还包了一层 `VariableState`**，所以路径会长成 `['mlps'][0]['backbone1']['kernel'].value` 这样——`.value` 是 `VariableState` 的字段名，不是模型里的属性。

In [3]:
graphdef, state = nnx.split(model)

print("graphdef 类型 :", type(graphdef).__name__)
print("state    类型 :", type(state).__name__)

leaves_with_path, treedef = jax.tree_util.tree_flatten_with_path(state)
print("叶子数        :", len(leaves_with_path))
for path, leaf in leaves_with_path[:4]:
    print(f"  {jax.tree_util.keystr(path):55s} {np.shape(leaf)!s:12s} {leaf.dtype}")
print("  …")
print("treedef       :", str(treedef)[:90], "…  (共", treedef.num_nodes, "个节点)")

graphdef 类型 : GraphDef
state    类型 : State
叶子数        : 24
  ['mlps'][0]['backbone1']['bias'].value                  (8,)         float64
  ['mlps'][0]['backbone1']['kernel'].value                (4, 8)       float64
  ['mlps'][0]['backbone2']['bias'].value                  (8,)         float64
  ['mlps'][0]['backbone2']['kernel'].value                (8, 8)       float64
  …
treedef       : PyTreeDef(CustomNode(State[('mlps',)], [{0: {'backbone1': {'bias': CustomNode(Param[(('eag …  (共 65 个节点)


## 3. 存成 npz

三个要点：

1. 叶子按统一编号 `p0000…` 命名，避免依赖原来的键名；
2. `__order__` 记录每个叶子的路径**（同时固定了展平顺序）**；
3. `__treedef__` 用 pickle 存 `PyTreeDef`。

`np.savez` 会用 `np.asarray` 把 jax 数组转成 numpy，不需要手动转换。

In [4]:
def save_params_npz(params, npz_path):
    """把 nnx.State（或任意 pytree）逐叶子存进 npz，返回叶子数。"""
    leaves_with_path, treedef = jax.tree_util.tree_flatten_with_path(params)
    arrays = {f"p{i:04d}": np.asarray(leaf) for i, (_, leaf) in enumerate(leaves_with_path)}
    # 固定展平顺序（npz 按键名排序，不保存顺序会错位）
    arrays["__order__"] = np.array(
        [jax.tree_util.keystr(p) for p, _ in leaves_with_path], dtype=str
    )
    # 保存结构；只存结构不存数值，体积很小
    arrays["__treedef__"] = np.frombuffer(pickle.dumps(treedef), dtype=np.uint8)
    np.savez(npz_path, **arrays)
    return len(leaves_with_path)


n_leaf = save_params_npz(state, "demo_params.npz")
print("已保存叶子数:", n_leaf)

import os
print("文件大小    :", os.path.getsize("demo_params.npz"), "B")

已保存叶子数: 24
文件大小    : 14911 B


看一眼 npz 里到底装了什么——注意 **键是按名字排序的**，`p0001` 排在 `__order__` 前面，这正是必须自己记录顺序的原因。

In [5]:
z = np.load("demo_params.npz")
print("条目数:", len(z.files))
print("键名（已排序）:", z.files[:6], "…")
print()
print("p0000 :", z["p0000"].shape, z["p0000"].dtype)
print("__order__ 前 3 项:")
for s in z["__order__"][:3]:
    print("   ", s)
print("__treedef__ :", z["__treedef__"].shape, z["__treedef__"].dtype, "(pickle 字节)")

条目数: 26
键名（已排序）: ['p0000', 'p0001', 'p0002', 'p0003', 'p0004', 'p0005'] …

p0000 : (8,) float64
__order__ 前 3 项:
    ['mlps'][0]['backbone1']['bias'].value
    ['mlps'][0]['backbone1']['kernel'].value
    ['mlps'][0]['backbone2']['bias'].value
__treedef__ : (1617,) uint8 (pickle 字节)


## 4. 读回并 `nnx.merge` 进模型

顺序是：`__treedef__` 反序列化 → 按 `p0000…` 取出叶子、用 `tree_unflatten` 重建 `State` → `nnx.merge(graphdef, state)` 得到可直接调用的模型。

In [6]:
def load_params_npz(npz_path):
    """与 save_params_npz 配对：读回参数树（State），结构与 dtype/shape 一致。"""
    z = np.load(npz_path, allow_pickle=False)   # 数据部分不需要 pickle
    treedef = pickle.loads(z["__treedef__"].tobytes())
    leaves = [z[f"p{i:04d}"] for i in range(len(z["__order__"]))]
    return jax.tree_util.tree_unflatten(treedef, leaves)


state_loaded = load_params_npz("demo_params.npz")
print("重建类型 :", type(state_loaded).__name__)

model_loaded = nnx.merge(graphdef, state_loaded)
y_loaded = model_loaded(x)

print("原模型输出 :", np.asarray(y_ref))
print("加载后输出 :", np.asarray(y_loaded))
print("输出完全相同:", bool(jnp.all(y_ref == y_loaded)))

重建类型 : State
原模型输出 : [[-0.24103521 -0.17351907  0.38299947]
 [-0.18949238 -0.14880035  0.36389023]
 [-0.08921869 -0.04138292  0.17255699]
 [ 0.09868382  0.03211686 -0.09456598]
 [ 0.20680205  0.03282466 -0.21687871]]
加载后输出 : [[-0.24103521 -0.17351907  0.38299947]
 [-0.18949238 -0.14880035  0.36389023]
 [-0.08921869 -0.04138292  0.17255699]
 [ 0.09868382  0.03211686 -0.09456598]
 [ 0.20680205  0.03282466 -0.21687871]]
输出完全相同: True


In [7]:
# 逐叶子比对：数量 / 路径 / dtype / 数值
pl, _ = jax.tree_util.tree_flatten_with_path(state)
ql, _ = jax.tree_util.tree_flatten_with_path(state_loaded)

print("叶子数一致   :", len(pl) == len(ql))
print("路径一致     :", all(jax.tree_util.keystr(a) == jax.tree_util.keystr(b) for (a, _), (b, _) in zip(pl, ql)))
print("dtype 全一致 :", all(np.asarray(u).dtype == np.asarray(v).dtype for (_, u), (_, v) in zip(pl, ql)))
print("最大元素差   :", max(float(np.max(np.abs(np.asarray(u) - np.asarray(v)))) for (_, u), (_, v) in zip(pl, ql)))

叶子数一致   : True
路径一致     : True
dtype 全一致 : True
最大元素差   : 0.0


## 5. 为什么必须带 `__order__` 和 `__treedef__`

两个反例，说明这两项元信息各自解决什么问题。

In [8]:
z2 = np.load("demo_params.npz")

# (1) npz 会按键名排序：'__order__' 以 '_' 开头，排在 'p0000' 之前
print("z.files 顺序 :", z2.files[:3], "…")
print("展平时的顺序 : p0000, p0001, p0002, …")
print("'__order__' 在 z.files 里的下标 :", z2.files.index("__order__"))

# 若直接把 z.files 当成叶子序列 → 叶子数量/类型对不上，unflatten 直接失败
try:
    jax.tree_util.tree_unflatten(treedef, [z2[k] for k in z2.files])
except Exception as e:
    print("按 z.files 直接反序列化 → 报错:", type(e).__name__, "|", str(e)[:100])

z.files 顺序 : ['p0000', 'p0001', 'p0002'] …
展平时的顺序 : p0000, p0001, p0002, …
'__order__' 在 z.files 里的下标 : 24
按 z.files 直接反序列化 → 报错: ValueError | Too many leaves for PyTreeDef; expected 24.


In [9]:
# (2) 键名不补零时，npz 的字典序会把 p10 排到 p2 前面 → 顺序彻底乱掉
naive = {f"p{i}": np.asarray(v) for i, (_, v) in enumerate(leaves_with_path)}   # 故意不补零
order = sorted(naive)
print("npz 读回顺序（前 12）:", order[:12])
print("对应的原展平下标      :", [int(k[1:]) for k in order[:12]])

try:
    st_bad = jax.tree_util.tree_unflatten(treedef, [naive[k] for k in order])
    y_bad = nnx.merge(graphdef, st_bad)(x)
    print("按排序键还原 → 输出与原模型相同:", bool(jnp.all(y_bad == y_ref)))
except Exception as e:
    print("按排序键还原 → 失败:", type(e).__name__, "|", str(e)[:110])

print()
print("所以键名一定要零填充（p0000）——或者，更稳妥地，显式记录 __order__（本实现两者都做了）。")

npz 读回顺序（前 12）: ['p0', 'p1', 'p10', 'p11', 'p12', 'p13', 'p14', 'p15', 'p16', 'p17', 'p18', 'p19']
对应的原展平下标      : [0, 1, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19]
按排序键还原 → 失败: TypeError | dot_general requires contracting dimensions to have the same shape, got (8,) and (1,).

所以键名一定要零填充（p0000）——或者，更稳妥地，显式记录 __order__（本实现两者都做了）。


In [10]:
# (3) 没有 treedef 就只能手工重建容器结构（要知道是 State、哪些键是 nnx.List 的整数下标…）
print("PyTreeDef 才能告诉 tree_unflatten 容器层级：")
print("  节点数 =", treedef.num_nodes, "| 叶子数 =", treedef.num_leaves)
print("因此 save_params_npz 里 pickle 的只是这棵“空骨架”，不含任何数值。")

PyTreeDef 才能告诉 tree_unflatten 容器层级：
  节点数 = 65 | 叶子数 = 24
因此 save_params_npz 里 pickle 的只是这棵“空骨架”，不含任何数值。


## 6. 加载进一个「重新初始化」的同类模型

这才是实际用途：模型重新随机初始化（或换了进程），把上次训练好的参数灌进去。

In [11]:
model2 = Ensemble(K=3, n_in=4, hidden=8, rngs=nnx.Rngs(12345))   # 不同种子 → 不同初始化
y_before = model2(x)
print("加载前（新初始化）输出:", np.asarray(y_before))

graphdef2, state2 = nnx.split(model2)
print("结构是否相同:", graphdef2 == graphdef)   # 结构一致才允许用同一份参数

model2 = nnx.merge(graphdef2, load_params_npz("demo_params.npz"))
y_after = model2(x)
print("加载后输出            :", np.asarray(y_after))
print("与原模型完全相同      :", bool(jnp.all(y_after == y_ref)))

加载前（新初始化）输出: [[ 0.44727322  0.19782353 -0.01620529]
 [ 0.18572373  0.17004467  0.03756648]
 [-0.10509306  0.05999403 -0.00290508]
 [-0.34617693 -0.03076019 -0.00994931]
 [-0.52132605 -0.10787239  0.09766612]]
结构是否相同: True
加载后输出            : [[-0.24103521 -0.17351907  0.38299947]
 [-0.18949238 -0.14880035  0.36389023]
 [-0.08921869 -0.04138292  0.17255699]
 [ 0.09868382  0.03211686 -0.09456598]
 [ 0.20680205  0.03282466 -0.21687871]]
与原模型完全相同      : True


## 7. 在真实脚本里怎么用

本仓库的训练脚本已经内置了这对函数（`NES-VMC K4/…K4_Ansatz_A_RealFFNN.py`、`NES-VMC K6/…K6_Ansatz_A_RealFFNN.py`）：

```python
# 训练结束
n_leaf = save_params_npz(total_params, PARAMS_NPZ)      # total_params 是 nnx.State

# 之后（任意进程）
jax.config.update("jax_enable_x64", True)               # 必须先开 x64
resp = load_params_npz(PARAMS_NPZ)                      # 取回 State
model = nnx.merge(graphdef, resp)                       # graphdef 来自 nnx.split(同结构模型)
psi   = model(x)
```

文件名建议用 `…_params_….npz` 与 `…_history_….pkl` 区分：pkl 存训练历史（含样本），npz 只存最终参数。

## 8. 几个坑

1. **先开 x64**：`jax.config.update("jax_enable_x64", True)`。JAX 在 x64 关闭时会把 float64 静默降成 float32——包括**反序列化 pickle 中的 jax 数组**。如果拿 `.pkl` 里的参数与 `.npz` 比对时忘了开 x64，会看到 ~1e-8 的“假差异”（我实测最大 3e-8，开启后严格为 0）。
2. **`allow_pickle=False`**：`.npz` 里的参数值本身是纯二进制数组，读的时候不需要 pickle；只有 `__treedef__` 那一个条目需要 `pickle.loads`。所以即使不信任来源的 pickle，参数本身仍可用 `numpy` 安全读取。
3. **npz 键会排序**：不要让“键名顺序”承担语义，务必自己存 `__order__`。
4. **只对 pytree 有效**：`state` 里所有叶子都必须是数组。若模型带非数组状态（例如 Python 标量、字符串），需要先转换成数组或另行保存。
5. **结构必须一致**：`nnx.merge` 要求 `graphdef` 与 `state` 结构匹配；改过网络结构（层数、维度、变量名）后旧 npz 不能直接灌进去，会报 path/shape 不匹配。
6. **大模型注意体积**：`np.savez` 是**非压缩**的。若参数很大，用 `np.savez_compressed` 即可（读法完全相同）。

In [12]:
# 清理演示文件（保留代码，不留在磁盘上）
os.remove("demo_params.npz")
print("已删除 demo_params.npz")

已删除 demo_params.npz
